# Clean up Data suitable for DMD Analysis

> 2024-04-01 
>
> 2024-04-19 

Starting with a csv file of all data, create numpy files (`.npz`) which contains subsetted data and just the data needed for DMD analysis.

The csv files were created `2024-03-24` using matlab, though does not contain anything new should be easier to subset. 

## Loading the Full Data



In [8]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format='retina'

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from BirdDMD import replace_rot_xyz_name, get_flight_modes, subset_by, get_column_names



unilateral_data = pd.read_csv("../data/raw/2024-03-24-FullUnilateralMarkers.csv")
bilateral_data = pd.read_csv("../data/raw/2024-03-24-FullBilateralMarkers.csv")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Clean up Column Names

In [2]:


unilateral_data = replace_rot_xyz_name(unilateral_data)
bilateral_data = replace_rot_xyz_name(bilateral_data)

bilateral_data.columns

Index(['frameID', 'seqID', 'time', 'HorzDistance', 'VertDistance',
       'body_pitch', 'BirdID', 'PerchDistance', 'Year', 'Naive', 'Obstacle',
       'IMU', 'left_wingtip_x', 'left_wingtip_y', 'left_wingtip_z',
       'right_wingtip_x', 'right_wingtip_y', 'right_wingtip_z',
       'left_primary_x', 'left_primary_y', 'left_primary_z', 'right_primary_x',
       'right_primary_y', 'right_primary_z', 'left_secondary_x',
       'left_secondary_y', 'left_secondary_z', 'right_secondary_x',
       'right_secondary_y', 'right_secondary_z', 'left_tailtip_x',
       'left_tailtip_y', 'left_tailtip_z', 'right_tailtip_x',
       'right_tailtip_y', 'right_tailtip_z'],
      dtype='object')

## Define Flight Behaviours

Uses the distance to the perch and time from starting perch to estimate the flight behaviours. These include:
- Initial: first wingbeat after takeoff
- Flapping: All the wingbeats
- Gliding: after flapping and before landing
- Landing: last metre before the perch

In [5]:
unilateral_data = get_flight_modes(unilateral_data)
bilateral_data = get_flight_modes(bilateral_data)

print(unilateral_data['behaviour'].unique())

['flapping_initial' 'flapping' '' 'gliding' 'landing']


## Save the data 

Saving csv files with just the flapping and initial behaviours for reference. 

In [21]:
subset = bilateral_data[bilateral_data['behaviour'].str.contains("flapping")]
subset.to_csv("../data/processed/BilateralFlapping.csv")

subset = unilateral_data[unilateral_data['behaviour'].str.contains("flapping")]
subset.to_csv("../data/processed/UnilateralFlapping.csv")


subset = bilateral_data[bilateral_data['behaviour'].str.contains("initial")]
subset.to_csv("../data/processed/BilateralInitial.csv")

subset = unilateral_data[unilateral_data['behaviour'].str.contains("initial")]
subset.to_csv("../data/processed/UnilateralInitial.csv")



# Saving numpy data for DMD input

The DMD analysis will be done on the different behaviours separately. This will be saved as a numpy npz file. As numpy does not have column names, these are saved in a separate file.

In [20]:

# Save the column names
marker_column_names, info_column_names = get_column_names(bilateral_data)
np.savez("../data/samples/ColumnNames.npz", marker_column_names = marker_column_names, info_column_names = info_column_names)


marker_data, info_data = subset_by(unilateral_data, 
                                   behaviour='flapping', 
                                   bird="Toothless", 
                                   perchDist=12)
np.savez("../data/samples/Flapping_12mToothless_Unilateral.npz", marker_data = marker_data, info_data = info_data)


marker_data, info_data = subset_by(bilateral_data, 
                                   behaviour='flapping', 
                                   bird='Toothless', 
                                   perchDist=12)
np.savez("../data/samples/Flapping_12mToothless_Bilateral.npz", marker_data = marker_data, info_data = info_data)


marker_data, info_data = subset_by(unilateral_data, 
                                   behaviour='initial', 
                                   bird='Toothless', 
                                   perchDist=12)
np.savez("../data/samples/Initial_12mToothless_Unilateral.npz", marker_data = marker_data, info_data = info_data)


marker_data, info_data = subset_by(bilateral_data, 
                                   behaviour='initial', 
                                   bird='Toothless', 
                                   perchDist=12)
np.savez("../data/samples/Initial_12mToothless_Bilateral.npz", marker_data = marker_data, info_data = info_data)



marker_data, info_data = subset_by(unilateral_data, 
                                   behaviour='initial', 
                                   bird='Toothless', 
                                   perchDist=9)
np.savez("../data/samples/Initial_9mToothless_Unilateral.npz", marker_data = marker_data, info_data = info_data)


marker_data, info_data = subset_by(bilateral_data, 
                                   behaviour='initial', 
                                   bird='Toothless', 
                                   perchDist=9)
np.savez("../data/samples/Initial_9mToothless_Bilateral.npz", marker_data = marker_data, info_data = info_data)


